[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/simple-graph.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58238187-lesson-2-simple-graph)

# 最简单的图

让我们构建一个包含 3 个节点和一条条件边的简单图。

![Screenshot 2024-08-20 at 3.11.22 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dba5f465f6e9a2482ad935_simple-graph1.png)

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph

## State（状态）

首先，定义图的 [State](https://docs.langchain.com/oss/python/langgraph/graph-api#state)。

State Schema 作为图中所有节点和边的输入 Schema。

我们使用 Python `typing` 模块中的 `TypedDict` 类作为 Schema，它为各个键提供类型提示。

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    graph_state: str

## Nodes（节点）

[Nodes](https://docs.langchain.com/oss/python/langgraph/graph-api/#nodes) 就是普通的 Python 函数。

第一个位置参数是上面定义的状态。

由于状态是一个具有上述 Schema 的 `TypedDict`，每个节点可以通过 `state['graph_state']` 访问键 `graph_state`。

每个节点返回状态键 `graph_state` 的新值。

默认情况下，每个节点返回的新值[会覆盖](https://docs.langchain.com/oss/python/langgraph/graph-api/#reducers)之前的状态值。

In [ ]:
def node_1(state):
    print("---Node 1---")
    return {"graph_state": state['graph_state'] +" I am"}

def node_2(state):
    print("---Node 2---")
    return {"graph_state": state['graph_state'] +" happy!"}

def node_3(state):
    print("---Node 3---")
    return {"graph_state": state['graph_state'] +" sad!"}

## Edges（边）

[Edges](https://docs.langchain.com/oss/python/langgraph/graph-api/#edges) 连接节点。

普通边用于*始终*从一个节点到另一个节点，例如从 `node_1` 到 `node_2`。

[条件边](https://docs.langchain.com/oss/python/langgraph/graph-api/#conditional-edges)用于*可选地*在节点之间路由。

条件边实现为根据某些逻辑返回下一个要访问的节点的函数。

In [ ]:
import random
from typing import Literal

def decide_mood(state) -> Literal["node_2", "node_3"]:
    
    # 通常，我们会使用状态来决定下一个要访问的节点
    user_input = state['graph_state'] 
    
    # 这里，我们简单地在节点 2 和 3 之间做 50/50 的随机选择
    if random.random() < 0.5:

        # 50% 的概率返回 Node 2
        return "node_2"
    
    # 50% 的概率返回 Node 3
    return "node_3"

## 图的构建

现在，我们使用上面定义的组件来构建图。

[StateGraph 类](https://docs.langchain.com/oss/python/langgraph/graph-api/#stategraph)是我们可以使用的图类。

首先，我们使用上面定义的 `State` 类初始化一个 StateGraph。

然后添加节点和边。

我们使用 [`START` 节点（一个特殊节点）](https://docs.langchain.com/oss/python/langgraph/graph-api/#start-node) 来指示图的起始位置，该节点将用户输入发送到图中。

[`END` 节点](https://docs.langchain.com/oss/python/langgraph/graph-api/#end-node)是一个表示终止节点的特殊节点。

最后，我们[编译图](https://docs.langchain.com/oss/python/langgraph/graph-api/#compiling-your-graph)以对图结构执行一些基本检查。

我们可以将图可视化为 [Mermaid 图](https://github.com/mermaid-js/mermaid)。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 编译
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

## 图的调用

编译后的图实现了 [runnable](https://reference.langchain.com/python/langchain_core/runnables/?h=runnables) 协议。

这提供了执行 LangChain 组件的标准方式。

`invoke` 是该接口中的标准方法之一。

输入是一个字典 `{"graph_state": "Hi, this is lance."}`，该字典设置了图状态字典的初始值。

当调用 `invoke` 时，图从 `START` 节点开始执行。

它按顺序遍历已定义的节点（`node_1`、`node_2`、`node_3`）。

条件边将使用 50/50 的决策规则从节点 `1` 走到节点 `2` 或 `3`。

每个节点函数接收当前状态并返回一个新值，该值会覆盖图状态。

执行持续进行，直到到达 `END` 节点。

In [ ]:
graph.invoke({"graph_state" : "Hi, this is Lance."})

`invoke` 同步运行整个图。

它会等待每一步完成后再进入下一步。

它返回所有节点执行完毕后图的最终状态。

在这个例子中，它返回 `node_3` 完成后的状态：

```
{'graph_state': 'Hi, this is Lance. I am sad!'}
```